# Week 2 — Character-Level N-gram Language Model

In this notebook, we will:

1. Load the dataset
2. Create a character tokenizer
3. Create Bigrams, Trigrams, and 4-grams
4. Calculate N-gram frequencies
5. Calculate conditional probabilities
6. Perform text generation
7. Compare Bigrams, Trigrams, and 4-grams
8. Compare greedy and random sampling

In [4]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import load_text_file, clean_dataset
from src.tokenizer import ArabicCharacterTokenizer
from src.ngram import CharacterNGramModel
from src.generator import NGramTextGenerator

In [5]:
dataset_path = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "arabic_sample.txt"
)

dataset = load_text_file(dataset_path)
dataset = clean_dataset(dataset)

print(f"Number of texts: {len(dataset)}")
print()
print(dataset[0])

Number of texts: 12

في يوم من الايام كان هناك رجل يعيش في مدينة صغيرة.


In [6]:
tokenizer = ArabicCharacterTokenizer()
tokenizer.fit(dataset)

print(f"Vocabulary size: {tokenizer.vocabulary_size}")
print()
print(tokenizer.vocabulary[:50])

Vocabulary size: 34

['<PAD>', ' ', '.', '،', 'ء', 'ئ', 'ا', 'ب', 'ة', 'ت', 'ث', 'ج', 'ح', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ع', 'غ', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'ه', 'و', 'ي', '<UNK>']


In [7]:
text = dataset[0]

tokens = tokenizer.character_tokenize(text)

print("Original:")
print(text)

print()
print("Character tokens:")
print(tokens[:50])

Original:
في يوم من الايام كان هناك رجل يعيش في مدينة صغيرة.

Character tokens:
['ف', 'ي', ' ', 'ي', 'و', 'م', ' ', 'م', 'ن', ' ', 'ا', 'ل', 'ا', 'ي', 'ا', 'م', ' ', 'ك', 'ا', 'ن', ' ', 'ه', 'ن', 'ا', 'ك', ' ', 'ر', 'ج', 'ل', ' ', 'ي', 'ع', 'ي', 'ش', ' ', 'ف', 'ي', ' ', 'م', 'د', 'ي', 'ن', 'ة', ' ', 'ص', 'غ', 'ي', 'ر', 'ة', '.']


In [8]:
sample_text = "الشمس طلعت"

tokens = tokenizer.character_tokenize(sample_text)

for n in [2, 3, 4]:

    model = CharacterNGramModel(n=n)

    ngrams = model.generate_ngrams(tokens)

    print(f"\n{n}-gram:")
    print(ngrams[:10])


2-gram:
[('ا', 'ل'), ('ل', 'ش'), ('ش', 'م'), ('م', 'س'), ('س', ' '), (' ', 'ط'), ('ط', 'ل'), ('ل', 'ع'), ('ع', 'ت')]

3-gram:
[('ا', 'ل', 'ش'), ('ل', 'ش', 'م'), ('ش', 'م', 'س'), ('م', 'س', ' '), ('س', ' ', 'ط'), (' ', 'ط', 'ل'), ('ط', 'ل', 'ع'), ('ل', 'ع', 'ت')]

4-gram:
[('ا', 'ل', 'ش', 'م'), ('ل', 'ش', 'م', 'س'), ('ش', 'م', 'س', ' '), ('م', 'س', ' ', 'ط'), ('س', ' ', 'ط', 'ل'), (' ', 'ط', 'ل', 'ع'), ('ط', 'ل', 'ع', 'ت')]


In [9]:
bigram_model = CharacterNGramModel(n=2)

bigram_model.fit(
    dataset,
    tokenizer,
)

print(bigram_model)

CharacterNGramModel(n=2, contexts=31)


In [10]:
trigram_model = CharacterNGramModel(n=3)

trigram_model.fit(
    dataset,
    tokenizer,
)

print(trigram_model)

CharacterNGramModel(n=3, contexts=190)


In [11]:
fourgram_model = CharacterNGramModel(n=4)

fourgram_model.fit(
    dataset,
    tokenizer,
)

print(fourgram_model)

CharacterNGramModel(n=4, contexts=321)


In [12]:
context = "ال"

print("Bigram counts:")
print(bigram_model.counts.get(context, {}))

print()
print("Trigram counts:")
print(trigram_model.counts.get(context, {}))

Bigram counts:
{}

Trigram counts:
Counter({'ر': 5, 'م': 4, 'ط': 4, 'ي': 3, 'ن': 2, 'س': 2, 'ش': 2, 'ب': 2, 'ا': 1, 'ص': 1, 'ه': 1, ' ': 1, 'ك': 1})


In [13]:
context = "ال"

probabilities = trigram_model.get_next_token_probabilities(
    context
)

print(f"Context: {context}")
print()

for token, probability in sorted(
    probabilities.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]:

    print(
        f"{repr(token)} -> "
        f"{probability:.4f}"
    )

Context: ال

'ر' -> 0.1724
'م' -> 0.1379
'ط' -> 0.1379
'ي' -> 0.1034
'ن' -> 0.0690
'س' -> 0.0690
'ش' -> 0.0690
'ب' -> 0.0690
'ا' -> 0.0345
'ص' -> 0.0345


In [14]:
total = sum(probabilities.values())

print(f"Total probability: {total:.4f}")

Total probability: 1.0000


In [15]:
context = "ال"

next_token = trigram_model.most_likely_next_token(
    context
)

print(
    f"Context: {context}"
)

print(
    f"Most likely next character: {next_token}"
)

Context: ال
Most likely next character: ر


In [16]:
prompt = "يوم واحد"

generator = NGramTextGenerator(
    fourgram_model,
    tokenizer,
)

generated_text = generator.generate(
    start_prompt=prompt,
    n_tokens=50,
    sampling_mode="greedy",
)

print(generated_text)

يوم واحد                                                  


In [17]:
generated_text = generator.generate(
    start_prompt=prompt,
    n_tokens=50,
    sampling_mode="random",
)

print(generated_text)

يوم واحد                                                  


In [22]:
models = {
    "Bigram": bigram_model,
    "Trigram": trigram_model,
    "4-gram": fourgram_model,
}

prompt = "يوم واحد"

for name, model in models.items():

    generator = NGramTextGenerator(
        model,
        tokenizer,
    )

    generated = generator.generate(
        start_prompt=prompt,
        n_tokens=50,
        sampling_mode="random",
    )

    print(name)
    print(generated)
    print()

Bigram
يوم واحد م. ادوده م ا صغيرب انه وه فيرالبان من امن اة، ي ف

Trigram
يوم واحد كان البحث حياة الا يبحث عنه الطريق.              

4-gram
يوم واحد                                                  



In [23]:
for name, model in models.items():

    generator = NGramTextGenerator(
        model,
        tokenizer,
    )

    generated = generator.generate(
        start_prompt=prompt,
        n_tokens=50,
        sampling_mode="greedy",
    )

    print(name)
    print(generated)
    print()

Bigram
يوم واحد ال ال ال ال ال ال ال ال ال ال ال ال ال ال ال ال ا

Trigram
يوم واحد الرجل الرجل الرجل الرجل الرجل الرجل الرجل الرجل ا

4-gram
يوم واحد                                                  



In [20]:
print("Model comparison")
print()

for name, model in models.items():

    print(
        f"{name}:"
    )

    print(
        f"  N-gram size : {model.n}"
    )

    print(
        f"  Contexts    : {len(model)}"
    )

    print(
        f"  Vocabulary  : {len(model.get_vocabulary())}"
    )

    print()

Model comparison

Bigram:
  N-gram size : 2
  Contexts    : 31
  Vocabulary  : 32

Trigram:
  N-gram size : 3
  Contexts    : 190
  Vocabulary  : 32

4-gram:
  N-gram size : 4
  Contexts    : 321
  Vocabulary  : 32

